# 基于HCCL的分布式字符串词频统计

字符串词频统计是文本分析、日志处理和自然语言预处理中的基础任务。单机环境可以直接扫描完整文本并更新哈希表，但在多设备环境中，输入文本通常被切分到不同进程，每个进程只能得到局部统计结果。为了使用集合通信汇总这些结果，需要先把变长字符串转换为结构一致、长度固定的数值向量。

本实验基于C++17、AscendCL和HCCL实现分布式字符串词频统计。程序首先构造固定字符表，将完整文本按Rank切分，并把每个Rank的局部文本统计为`int64`计数向量；随后通过ReduceScatter完成逐元素全局求和与结果分片，再通过AllGather收集各Rank分片，使每个Rank恢复完整的全局计数向量。最后，程序将通信结果与CPU串行参考结果逐元素比较，并输出全局Top-K字符、通信耗时和缓冲区规模。

本节学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建实验目录、加载CANN环境并写入公共接口；
3. 问题分析：分析词表、计数向量、文本分片、向量补齐和集合通信流程；
4. 核心程序开发：实现语料生成、本地统计、模拟集合通信、HCCL后端和结果报告；
5. 结果验证与性能分析：完成工程构建、双Rank运行、正确性校验和通信指标分析；
6. 实验总结：归纳从变长字符串到定长通信缓冲区的完整实现过程。


---
## 1. 实验概述

本实验以分布式字符串词频统计为背景，研究变长文本如何转换为适合集合通信处理的定长数据结构。实验同时提供`simulate`和`hccl`两种后端：`simulate`后端在单进程中模拟多个Rank，用于验证数据组织和通信语义；`hccl`后端在两张昇腾NPU上启动两个进程，执行真实的ReduceScatter和AllGather集合通信。


### 1.1 实验目标

完成本实验后应达到以下目标：

1. 理解分布式字符串词频统计中的数据结构。能够说明字符表、字符到Token编号的映射、局部计数向量、补齐后计数向量和结果分片之间的组织关系，理解变长文本转换为定长`int64`向量的必要性。
2. 掌握分布式词频统计的核心算法和通信流程。能够完成文本生成与Rank切分、本地词频统计、向量补齐，理解ReduceScatter的逐元素归约与结果分片过程，以及AllGather收集分片并恢复完整全局计数向量的过程。
3. 具备结果验证和性能分析能力。能够使用CPU串行统计结果验证HCCL汇总结果，结合`mismatch_count`、全局Top-K字符、通信缓冲区规模、ReduceScatter耗时和AllGather耗时分析实验结果。


### 1.2 前置知识

本实验要求提前具备以下基础：

1. 字符串与数组基础：理解字符串遍历、字符编码、数组下标和哈希映射的基本使用方法。
2. 分布式进程基础：理解`world_size`表示通信域中的总Rank数，`rank`表示当前进程编号，`local_device`表示当前进程绑定的本地NPU编号。
3. 集合通信基础：理解Reduce、Scatter和AllGather的基本语义，能够区分局部统计结果、规约后结果分片和完整全局结果。
4. C++与CMake基础：能够阅读C++17工程，了解头文件、源文件、CMake构建配置和Shell运行脚本之间的关系。
5. AscendCL与HCCL基础：了解Host内存、Device内存、执行流、通信域和设备同步的作用。


### 1.3 实验要点

实验中应重点关注以下内容：

1. 数据表示：使用`Vocabulary`建立字符与Token编号的稳定映射，将局部字符串转换为定长计数向量；
2. 数据切分：使用相同随机种子生成全局文本，并按连续区间切分给不同Rank，保证各进程获得一致的全局数据视图；
3. 长度对齐：将词表长度补齐到`world_size`的整数倍，保证ReduceScatter可以向每个Rank分发等长分片；
4. 集合通信：使用ReduceScatter完成逐元素求和和结果切分，再使用AllGather恢复完整全局计数向量；
5. 结果分析：使用串行参考结果、Top-K字符、缓冲区字节数和通信耗时验证程序正确性并观察运行特征。


---
## 2. 环境准备

### 2.1 创建实验目录并加载CANN环境

本小节创建实验所需目录，并尝试加载Ascend CANN环境变量。

目录划分如下：

- `src/03.01_inclass_hccl_string_statistics/include`：保存实验配置、词表、统计和通信接口；
- `src/03.01_inclass_hccl_string_statistics/src`：保存语料生成、本地统计、集合通信和结果报告实现；
- `src/03.01_inclass_hccl_string_statistics/scripts`：保存模拟后端和真实HCCL后端运行脚本；
- `src/03.01_inclass_hccl_string_statistics/results`：保存实验输出和各Rank日志。

如果当前环境没有安装CANN，仍然可以生成完整工程并运行`simulate`后端；真实双Rank HCCL实验需要在已安装CANN且至少有两张可见NPU的服务器上执行。


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

WORK_DIR = Path("src/03.01_inclass_hccl_string_statistics").resolve()
SRC_DIR = WORK_DIR / "src"

# 创建实验目录结构
for directory in [WORK_DIR / "include", SRC_DIR, WORK_DIR / "scripts", WORK_DIR / "results"]:
    directory.mkdir(parents=True, exist_ok=True)

# 自动查找并加载CANN环境
arch = os.uname().machine
candidate_paths = []

for name in ["ASCEND_INSTALL_PATH", "ASCEND_TOOLKIT_HOME"]:
    value = os.environ.get(name)
    if value:
        candidate_paths.append(Path(value))

ascend_home = os.environ.get("ASCEND_HOME_PATH")
if ascend_home:
    candidate_paths.append(Path(ascend_home) / f"{arch}-linux")

candidate_paths.extend(sorted(Path("/opt/conda/Ascend").glob(f"cann-*/{arch}-linux"), reverse=True))
candidate_paths.append(Path("/usr/local/Ascend/ascend-toolkit/latest"))
candidate_paths.extend(
    sorted(Path("/usr/local/Ascend/ascend-toolkit").glob(f"*/{arch}-linux"), reverse=True)
)

set_env = None
for item in candidate_paths:
    if (item / "set_env.sh").exists():
        set_env = item / "set_env.sh"
        break

if set_env is not None:
    command = f"source {shlex.quote(str(set_env))} && env"
    env_text = subprocess.check_output(["bash", "-lc", command], text=True)
    for line in env_text.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    os.environ["ASCEND_TOOLKIT_HOME"] = str(set_env.parent)
    print("Ascend environment loaded from:", set_env)
else:
    print("Ascend set_env.sh was not found. The simulate backend can still run.")

print("Experiment directory:", WORK_DIR)
print("Source directory:", SRC_DIR)


### 2.2 写入工程公共接口

本节写入实验配置、语料、词表、本地统计、集合通信、HCCL后端和结果报告接口。第4节将继续写入这些接口对应的C++实现。运行到第5节后，`src/03.01_inclass_hccl_string_statistics`目录下将形成可直接编译运行的完整工程。


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/include/experiment_config.hpp
#pragma once

#include <cstdint>
#include <string>

namespace hccl_string_stats {

enum class Backend {
    // simulate 用于在单进程中模拟多个 rank，hccl 用于真实 HCCL 多进程运行。
    Simulate,
    Hccl,
};

struct ExperimentConfig {
    Backend backend = Backend::Simulate;
    // world_size 表示参与统计的逻辑 rank 数；HCCL 模式下通常等于启动的进程数。
    int world_size = 4;
    // rank 和 local_device 只在真实 HCCL 后端中使用，分别表示当前进程编号和本地设备号。
    int rank = 0;
    int local_device = 0;
    // corpus_mode 控制字符串数据分布，用于观察均匀、倾斜或 rank 偏置场景。
    std::string corpus_mode = "rank_biased";
    int total_length = 20000;
    std::uint32_t seed = 2026;
    int top_k = 12;
    // rank0 将 HCCL root info 写入该文件，其他 rank 读取后建立同一个通信域。
    std::string root_info_file = "/tmp/hccl_string_statistics_root.info";
};

Backend parse_backend(const std::string& value);
std::string backend_name(Backend backend);
void validate_config(const ExperimentConfig& config);

}  // namespace hccl_string_stats


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/include/vocab.hpp
#pragma once

#include <cstddef>
#include <string>
#include <unordered_map>
#include <vector>

namespace hccl_string_stats {

struct Vocabulary {
    std::vector<std::string> tokens;
    std::unordered_map<char, std::size_t> char_to_id;
    std::size_t unk_id = 0;
};

Vocabulary build_vocabulary();
std::string display_token(const std::string& token);

}  // namespace hccl_string_stats


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/include/corpus.hpp
#pragma once

#include <cstdint>
#include <string>
#include <vector>

namespace hccl_string_stats {

// 按指定分布生成完整字符串语料，所有 rank 都基于同一份语料做切分和校验。
std::string generate_corpus(const std::string& mode, int total_length, std::uint32_t seed);

// 将完整语料按 rank 数切成连续片段，模拟每个 rank 只持有自己的本地文本。
std::vector<std::string> split_text_by_rank(const std::string& text, int world_size);

}  // namespace hccl_string_stats


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/include/string_statistics.hpp
#pragma once

#include <cstddef>
#include <cstdint>
#include <string>
#include <utility>
#include <vector>

#include "vocab.hpp"

namespace hccl_string_stats {

using Count = std::int64_t;
using CountVector = std::vector<Count>;

struct RankLocalResult {
    int rank = 0;
    // 当前 rank 持有的本地字符串片段。
    std::string text;
    // 当前 rank 对本地字符串统计得到的词频向量。
    CountVector count_vector;
};

struct PaddingInfo {
    // 补齐后的词表长度，保证可以被 world_size 均分。
    std::size_t padded_vocab_size = 0;
    // ReduceScatter 后每个 rank 拿到的向量分片长度。
    std::size_t shard_size = 0;
};

struct BufferEstimate {
    // 每个 rank 输入完整计数向量需要的字节数。
    std::size_t input_count_vector_bytes_per_rank = 0;
    // ReduceScatter 后每个 rank 保留分片需要的字节数。
    std::size_t reduce_scatter_shard_bytes_per_rank = 0;
    // AllGather 后每个 rank 恢复完整向量需要的字节数。
    std::size_t all_gather_result_bytes_per_rank = 0;
    // 所有 rank 输入缓冲区的总字节数。
    std::size_t all_ranks_input_bytes_total = 0;
};

// 计算补齐长度和每个 rank 的分片长度。
PaddingInfo compute_padding(std::size_t vocab_size, int world_size);

// 统计单个 rank 本地字符串中的字符频次。
CountVector count_local_text(const std::string& text, const Vocabulary& vocabulary);

// 串行参考结果：直接扫描完整字符串，用于和通信结果做正确性对比。
CountVector serial_reference(const std::string& text, const Vocabulary& vocabulary);
CountVector pad_vector(const CountVector& vector, std::size_t padded_size);
CountVector unpad_vector(const CountVector& vector, std::size_t original_size);

// 比较全局通信结果和串行参考结果，返回是否一致以及不一致位置。
std::pair<bool, std::vector<std::size_t>> verify_result(
    const CountVector& global_count,
    const CountVector& reference_count);

// 按计数从高到低取 Top-K 字符，计数相同则按 token 字典序排序。
std::vector<std::pair<std::string, Count>> topk_tokens(
    const CountVector& count_vector,
    const Vocabulary& vocabulary,
    int top_k);

// 根据向量长度和 rank 数估算通信缓冲区规模。
BufferEstimate estimate_bytes(std::size_t padded_vocab_size, std::size_t shard_size, int world_size);

}  // namespace hccl_string_stats


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/include/collectives.hpp
#pragma once

#include <cstddef>
#include <vector>

#include "string_statistics.hpp"

namespace hccl_string_stats {

struct CommunicationResult {
    // 词表长度补齐到可被 world_size 整除，便于 ReduceScatter 等长切片。
    std::size_t padded_vocab_size = 0;
    // 每个 rank 在 ReduceScatter 后得到的分片长度。
    std::size_t shard_size = 0;
    // 每个 rank 持有的规约后分片，用于展示 ReduceScatter 的输出。
    std::vector<CountVector> reduced_shards;
    // AllGather 后恢复出的完整全局计数向量。
    CountVector gathered_count;
    double reduce_scatter_ms = 0.0;
    double all_gather_ms = 0.0;
};

// CPU 模拟版集合通信：先求和，再切片，最后拼回完整向量。
CommunicationResult run_simulated_collectives(
    const std::vector<CountVector>& local_count_vectors,
    int world_size);

}  // namespace hccl_string_stats


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/include/hccl_backend.hpp
#pragma once

#include <string>

#include "collectives.hpp"

namespace hccl_string_stats {

struct HcclRuntimeOptions {
    // 当前 HCCL 进程的 rank 编号。
    int rank = 0;
    // HCCL 通信域中的总 rank 数。
    int world_size = 1;
    // 当前进程绑定的本地 Ascend 设备号。
    int local_device = 0;
    // 用于在多个进程之间交换 HcclRootInfo 的临时文件。
    std::string root_info_file;
};

bool is_hccl_compiled();

// 真实 HCCL 后端：把本地计数向量放到设备内存，执行 ReduceScatter 和 AllGather。
CommunicationResult run_hccl_collectives(
    const CountVector& local_count,
    std::size_t vocab_size,
    const HcclRuntimeOptions& options);

}  // namespace hccl_string_stats


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/include/result_report.hpp
#pragma once

#include <string>
#include <vector>

#include "collectives.hpp"
#include "experiment_config.hpp"
#include "string_statistics.hpp"
#include "vocab.hpp"

namespace hccl_string_stats {

void print_experiment_config(
    const ExperimentConfig& config,
    const Vocabulary& vocabulary,
    const CommunicationResult& communication);
void print_rank_summary(const std::vector<RankLocalResult>& local_results, const Vocabulary& vocabulary);
void print_shard_summary(const std::vector<CountVector>& shards);
void print_topk(const std::vector<std::pair<std::string, Count>>& top_tokens);
void print_timing(const CommunicationResult& communication, Backend backend);
void print_buffer_estimate(const BufferEstimate& estimate);
void print_correctness(bool correct, const std::vector<std::size_t>& mismatch_indices, double load_imbalance);
void write_result_summary(
    const std::string& output_path,
    const ExperimentConfig& config,
    const CommunicationResult& communication,
    bool correct,
    const std::vector<std::size_t>& mismatch_indices,
    double load_imbalance,
    const std::vector<std::pair<std::string, Count>>& top_tokens);

}  // namespace hccl_string_stats


### 2.3 工程公共接口检查

公共接口写入完成后，检查关键头文件是否已经生成。检查结果均为`OK`时，说明实验配置、数据结构和模块接口已经就绪。


In [ ]:
from pathlib import Path

required_headers = [
    "include/experiment_config.hpp",
    "include/vocab.hpp",
    "include/corpus.hpp",
    "include/string_statistics.hpp",
    "include/collectives.hpp",
    "include/hccl_backend.hpp",
    "include/result_report.hpp",
]

for relative_path in required_headers:
    path = WORK_DIR / relative_path
    print(f"{relative_path}: {'OK' if path.exists() else 'MISSING'}")


---
## 3. 问题分析

本节分析分布式字符串词频统计的输入输出、词表与计数向量、文本切分、向量补齐、集合通信流程和默认实验参数。后续C++实现将围绕这些数据结构和流程展开。


### 3.1 输入输出与整体数据流

实验输入为一段长度为`total_length`的字符串，以及参与统计的Rank数量`world_size`。字符串包含小写字母、数字、空格和常用标点，程序支持`uniform`、`skewed`和`rank_biased`三种语料模式。

每个Rank输出一个局部字符计数向量。经过集合通信后，每个Rank得到完整的全局字符计数向量。整体数据流为：

```text
全局字符串
→ 按Rank切分本地文本
→ 本地字符计数向量
→ 向量补齐
→ ReduceScatter逐元素求和并切分
→ AllGather收集分片
→ 完整全局计数向量
→ CPU串行结果校验与Top-K分析
```


### 3.2 字符表与定长计数向量

`Vocabulary`包含三个核心成员：

- `tokens`：按照Token编号保存字符显示值；
- `char_to_id`：建立字符到Token编号的映射；
- `unk_id`：记录`<UNK>`对应的编号，用于统计词表外字符。

默认字符表包含26个小写字母、10个数字、8个空格或标点字符，并在末尾追加`<UNK>`，因此实际词表长度为45。计数向量中第$i$个元素表示`tokens[i]`的出现次数。无论本地文本长度和内容如何变化，各Rank都生成相同长度的向量，这使其能够直接作为集合通信缓冲区。


### 3.3 文本切分与本地统计

所有Rank使用相同的`corpus_mode`、`total_length`和`seed`生成同一份全局文本。程序按照下面的向上取整方式计算每个Rank的文本块长度：

$$
chunkSize=\left\lceil\frac{totalLength}{worldSize}\right\rceil
$$

第$r$个Rank负责的字符区间为：

$$
[r\times chunkSize,\min((r+1)\times chunkSize,totalLength))
$$

本地统计阶段逐字符扫描文本，将字符统一转换为小写，再通过`char_to_id`找到向量下标并累加计数。词表外字符统一计入`<UNK>`位置。


### 3.4 向量补齐与结果分片

ReduceScatter要求每个Rank接收相同数量的元素。设原始词表长度为$V$，Rank数量为$P$，则每个Rank的结果分片长度为：

$$
shardSize=\left\lceil\frac{V}{P}\right\rceil
$$

补齐后的向量长度为：

$$
paddedVocabSize=shardSize\times P
$$

本实验双Rank配置下，$V=45$、$P=2$，因此`shard_size=23`、`padded_vocab_size=46`。新增的补齐位置使用0填充，不会改变真实Token的计数结果。


### 3.5 ReduceScatter与AllGather

设两个Rank的补齐后局部计数向量分别为$x^{(0)}$和$x^{(1)}$。ReduceScatter首先逐元素求和：

$$
g_i=x_i^{(0)}+x_i^{(1)}
$$

然后把全局向量$g$切成两个等长分片。Rank 0获得前23个元素，Rank 1获得后23个元素。AllGather再按Rank顺序收集两个分片，使每个Rank恢复长度为46的完整向量。去掉末尾补齐位置后，即得到长度为45的全局词频向量。

ReduceScatter的输入是每个Rank持有的完整局部贡献向量，输出是该Rank负责的全局结果分片；AllGather的输入是本Rank分片，输出是由所有分片拼接形成的完整全局结果。


### 3.6 simulate与HCCL执行流程

`simulate`后端在一个进程中构造所有Rank的局部计数向量，使用CPU循环模拟逐元素求和、等长切片和结果拼接。它不依赖CANN和NPU，适合先验证数据结构与集合通信语义。

`hccl`后端为每个Rank启动一个独立进程，并将该进程绑定到对应NPU。Rank 0生成`HcclRootInfo`并写入临时文件，其他Rank读取该文件后加入同一个通信域。每个进程只把自己的局部计数向量复制到Device内存，再调用`HcclReduceScatter`和`HcclAllGather`完成真实设备间通信。


### 3.7 实验参数设置

本实验以双Rank真实HCCL运行为主要验证场景。默认生成长度为20000的`rank_biased`文本，随机种子为2026，输出全局Top-12字符。字符表长度为45，补齐后长度为46，每个Rank获得23个`int64`结果元素。


In [ ]:
import math

WORLD_SIZE = 2
TOTAL_LENGTH = 20000
VOCAB_SIZE = 45
COUNT_BYTES = 8
SEED = 2026
TOP_K = 12

SHARD_SIZE = math.ceil(VOCAB_SIZE / WORLD_SIZE)
PADDED_VOCAB_SIZE = SHARD_SIZE * WORLD_SIZE
INPUT_BYTES_PER_RANK = PADDED_VOCAB_SIZE * COUNT_BYTES
SHARD_BYTES_PER_RANK = SHARD_SIZE * COUNT_BYTES
ALL_GATHER_BYTES_PER_RANK = PADDED_VOCAB_SIZE * COUNT_BYTES

print("worldSize:", WORLD_SIZE)
print("totalLength:", TOTAL_LENGTH)
print("vocabSize:", VOCAB_SIZE)
print("paddedVocabSize:", PADDED_VOCAB_SIZE)
print("shardSize:", SHARD_SIZE)
print("input bytes per rank:", INPUT_BYTES_PER_RANK)
print("ReduceScatter shard bytes per rank:", SHARD_BYTES_PER_RANK)
print("AllGather result bytes per rank:", ALL_GATHER_BYTES_PER_RANK)


需要注意，当前通信缓冲区只有数百字节，通信时间更容易受到HCCL首次调用、Rank同步、设备调度和流同步等固定开销影响。因此，本实验的主要目标是验证数据组织和集合通信流程的正确性；若要形成严格的通信性能结论，应增加预热并重复运行多次，统计中位数、平均值和波动范围。


---
## 4. 核心程序开发

本节依次实现实验配置、字符表与语料、本地计数、模拟集合通信、真实HCCL后端、结果报告和主流程。所有实现均与交付版C++工程保持一致。


### 4.1 字符表与语料生成

`build_vocabulary`按照固定顺序建立字符表和字符到Token编号的映射，并在末尾追加`<UNK>`。`generate_corpus`使用固定随机种子生成可复现文本；`split_text_by_rank`把全局文本切分为连续片段，保证每个Rank只统计自己的本地数据。


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/src/vocab.cpp
#include "vocab.hpp"

#include <string>

namespace hccl_string_stats {

Vocabulary build_vocabulary() {
    Vocabulary vocabulary;
    // 实验把字符统计转化为固定长度向量统计，这里的字符顺序就是向量下标顺序。
    const std::string characters = "abcdefghijklmnopqrstuvwxyz0123456789 .,;:!?-";

    vocabulary.tokens.reserve(characters.size() + 1);
    for (char ch : characters) {
        const std::size_t index = vocabulary.tokens.size();
        vocabulary.tokens.emplace_back(1, ch);
        vocabulary.char_to_id[ch] = index;
    }

    vocabulary.unk_id = vocabulary.tokens.size();
    // 词表外字符统一落到 <UNK>，保证任意输入字符都能被统计。
    vocabulary.tokens.emplace_back("<UNK>");
    return vocabulary;
}

std::string display_token(const std::string& token) {
    if (token == " ") {
        return "' '";
    }
    return token;
}

}  // namespace hccl_string_stats


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/src/corpus.cpp
#include "corpus.hpp"

#include <algorithm>
#include <cmath>
#include <random>
#include <stdexcept>

namespace hccl_string_stats {
namespace {

char sample_char(const std::string& alphabet, std::mt19937& rng) {
    std::uniform_int_distribution<std::size_t> dist(0, alphabet.size() - 1);
    return alphabet[dist(rng)];
}

}  // namespace

std::string generate_corpus(const std::string& mode, int total_length, std::uint32_t seed) {
    if (total_length < 0) {
        throw std::invalid_argument("total_length must be non-negative");
    }

    std::mt19937 rng(seed);
    const std::string letters = "abcdefghijklmnopqrstuvwxyz";
    const std::string digits = "0123456789";
    const std::string punctuation = " .,;:!?-";

    std::string text;
    text.reserve(static_cast<std::size_t>(total_length));

    if (mode == "uniform") {
        // 均匀模式：字母、数字和标点基本等概率出现。
        const std::string alphabet = letters + digits + punctuation;
        for (int i = 0; i < total_length; ++i) {
            text.push_back(sample_char(alphabet, rng));
        }
        return text;
    }

    if (mode == "skewed") {
        // 倾斜模式：人为提高 e/t/a/o/n/r 等字符出现概率，模拟热点 token。
        const std::string weighted_chars =
            "eeeeeeeeeeeeeeeettttttttttttaaaaaaaaoooooooonnnnnnrrrrrr" +
            letters + digits + punctuation;
        for (int i = 0; i < total_length; ++i) {
            text.push_back(sample_char(weighted_chars, rng));
        }
        return text;
    }

    if (mode == "rank_biased") {
        // rank_biased 模式：周期性写入不同字符模式，让不同文本片段呈现局部偏置。
        const std::vector<std::string> patterns = {
            "aaaaabbbbccccddddeeee     ",
            "nnnnnoooooppppqqqqrrrr;;;;",
            "xxxxxyyyyyzzzzz1111122222!",
            "data-structure-hccl-rank-",
        };
        const std::string alphabet = letters + digits + punctuation;
        std::size_t pattern_index = 0;
        while (text.size() < static_cast<std::size_t>(total_length)) {
            text += patterns[pattern_index % patterns.size()];
            for (int i = 0; i < 16; ++i) {
                text.push_back(sample_char(alphabet, rng));
            }
            ++pattern_index;
        }
        text.resize(static_cast<std::size_t>(total_length));
        return text;
    }

    throw std::invalid_argument("unsupported corpus_mode: " + mode);
}

std::vector<std::string> split_text_by_rank(const std::string& text, int world_size) {
    if (world_size <= 0) {
        throw std::invalid_argument("world_size must be positive");
    }

    // 使用向上取整的 chunk_size，保证最后一段不足时也能保留所有字符。
    const std::size_t chunk_size =
        static_cast<std::size_t>(std::ceil(static_cast<double>(text.size()) / world_size));
    std::vector<std::string> pieces;
    pieces.reserve(static_cast<std::size_t>(world_size));

    for (int rank = 0; rank < world_size; ++rank) {
        const std::size_t start = static_cast<std::size_t>(rank) * chunk_size;
        const std::size_t end = std::min(start + chunk_size, text.size());
        if (start >= text.size()) {
            pieces.emplace_back();
        } else {
            pieces.emplace_back(text.substr(start, end - start));
        }
    }

    return pieces;
}

}  // namespace hccl_string_stats


### 4.2 实验配置与本地统计

实验配置模块负责解析`simulate`和`hccl`后端，并检查Rank、设备号、文本长度和Top-K等参数。本地统计模块负责计算补齐长度、生成局部计数向量、去除补齐位置、选择Top-K字符、估算通信缓冲区并执行正确性比较。


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/src/experiment_config.cpp
#include "experiment_config.hpp"

#include <stdexcept>

namespace hccl_string_stats {

Backend parse_backend(const std::string& value) {
    if (value == "simulate") {
        return Backend::Simulate;
    }
    if (value == "hccl") {
        return Backend::Hccl;
    }
    throw std::invalid_argument("unsupported backend: " + value);
}

std::string backend_name(Backend backend) {
    switch (backend) {
        case Backend::Simulate:
            return "simulate";
        case Backend::Hccl:
            return "hccl";
    }
    return "unknown";
}

void validate_config(const ExperimentConfig& config) {
    if (config.world_size <= 0) {
        throw std::invalid_argument("world_size must be positive");
    }
    if (config.rank < 0 || config.rank >= config.world_size) {
        throw std::invalid_argument("rank must be in [0, world_size)");
    }
    if (config.local_device < 0) {
        throw std::invalid_argument("local_device must be non-negative");
    }
    if (config.total_length < 0) {
        throw std::invalid_argument("total_length must be non-negative");
    }
    if (config.top_k < 0) {
        throw std::invalid_argument("top_k must be non-negative");
    }
}

}  // namespace hccl_string_stats


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/src/string_statistics.cpp
#include "string_statistics.hpp"

#include <algorithm>
#include <cctype>
#include <stdexcept>

namespace hccl_string_stats {

PaddingInfo compute_padding(std::size_t vocab_size, int world_size) {
    if (world_size <= 0) {
        throw std::invalid_argument("world_size must be positive");
    }
    const std::size_t ranks = static_cast<std::size_t>(world_size);
    // ReduceScatter 要求每个 rank 接收等长数据，因此先把词表长度补齐到 rank 数的倍数。
    const std::size_t shard_size = (vocab_size + ranks - 1) / ranks;
    return PaddingInfo{shard_size * ranks, shard_size};
}

CountVector count_local_text(const std::string& text, const Vocabulary& vocabulary) {
    CountVector counts(vocabulary.tokens.size(), 0);
    for (unsigned char raw_char : text) {
        // 统计时统一转为小写；词表外字符落到 <UNK>。
        const char lowered = static_cast<char>(std::tolower(raw_char));
        const auto found = vocabulary.char_to_id.find(lowered);
        const std::size_t token_id = found == vocabulary.char_to_id.end()
                                         ? vocabulary.unk_id
                                         : found->second;
        counts[token_id] += 1;
    }
    return counts;
}

CountVector serial_reference(const std::string& text, const Vocabulary& vocabulary) {
    return count_local_text(text, vocabulary);
}

CountVector pad_vector(const CountVector& vector, std::size_t padded_size) {
    if (vector.size() > padded_size) {
        throw std::invalid_argument("padded_size cannot be smaller than vector size");
    }
    // 补齐位置填 0，不影响真实 token 的计数结果。
    CountVector padded = vector;
    padded.resize(padded_size, 0);
    return padded;
}

CountVector unpad_vector(const CountVector& vector, std::size_t original_size) {
    if (vector.size() < original_size) {
        throw std::invalid_argument("original_size cannot be larger than vector size");
    }
    return CountVector(vector.begin(), vector.begin() + static_cast<std::ptrdiff_t>(original_size));
}

std::pair<bool, std::vector<std::size_t>> verify_result(
    const CountVector& global_count,
    const CountVector& reference_count) {
    const std::size_t compare_size = std::min(global_count.size(), reference_count.size());
    std::vector<std::size_t> mismatch_indices;

    // 逐位置比较通信恢复结果和串行参考结果；位置就是词表中的 token id。
    for (std::size_t index = 0; index < compare_size; ++index) {
        if (global_count[index] != reference_count[index]) {
            mismatch_indices.push_back(index);
        }
    }
    // 长度不一致时，多出来的位置也视为 mismatch。
    for (std::size_t index = compare_size; index < global_count.size(); ++index) {
        mismatch_indices.push_back(index);
    }
    for (std::size_t index = compare_size; index < reference_count.size(); ++index) {
        mismatch_indices.push_back(index);
    }

    return {mismatch_indices.empty(), mismatch_indices};
}

std::vector<std::pair<std::string, Count>> topk_tokens(
    const CountVector& count_vector,
    const Vocabulary& vocabulary,
    int top_k) {
    std::vector<std::pair<std::string, Count>> pairs;
    const std::size_t size = std::min(count_vector.size(), vocabulary.tokens.size());
    pairs.reserve(size);

    for (std::size_t index = 0; index < size; ++index) {
        pairs.push_back({vocabulary.tokens[index], count_vector[index]});
    }

    std::sort(pairs.begin(), pairs.end(), [](const auto& lhs, const auto& rhs) {
        if (lhs.second != rhs.second) {
            return lhs.second > rhs.second;
        }
        return lhs.first < rhs.first;
    });

    if (top_k >= 0 && pairs.size() > static_cast<std::size_t>(top_k)) {
        pairs.resize(static_cast<std::size_t>(top_k));
    }
    return pairs;
}

BufferEstimate estimate_bytes(std::size_t padded_vocab_size, std::size_t shard_size, int world_size) {
    const std::size_t element_size = sizeof(Count);
    const std::size_t ranks = static_cast<std::size_t>(world_size);
    // Count 使用 int64，因此字节数 = 元素个数 * sizeof(Count)。
    return BufferEstimate{
        padded_vocab_size * element_size,
        shard_size * element_size,
        padded_vocab_size * element_size,
        padded_vocab_size * element_size * ranks,
    };
}

}  // namespace hccl_string_stats


### 4.3 CPU模拟集合通信

模拟后端先检查所有Rank计数向量长度是否一致，再将向量补齐到`world_size`的整数倍。ReduceScatter模拟过程由“逐元素全局求和”和“按Rank等长切片”组成；AllGather模拟过程按Rank顺序拼接所有结果分片。


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/src/simulate_collectives.cpp
#include "collectives.hpp"

#include <chrono>
#include <stdexcept>

namespace hccl_string_stats {

CommunicationResult run_simulated_collectives(
    const std::vector<CountVector>& local_count_vectors,
    int world_size) {
    if (world_size <= 0) {
        throw std::invalid_argument("world_size must be positive");
    }
    if (local_count_vectors.empty()) {
        throw std::invalid_argument("local_count_vectors cannot be empty");
    }
    if (local_count_vectors.size() != static_cast<std::size_t>(world_size)) {
        throw std::invalid_argument("local_count_vectors size must equal world_size");
    }

    const std::size_t vocab_size = local_count_vectors.front().size();
    for (const auto& vector : local_count_vectors) {
        if (vector.size() != vocab_size) {
            throw std::invalid_argument("all count vectors must have the same size");
        }
    }

    const PaddingInfo padding = compute_padding(vocab_size, world_size);
    std::vector<CountVector> padded_vectors;
    padded_vectors.reserve(local_count_vectors.size());
    for (const auto& vector : local_count_vectors) {
        // 所有 rank 的输入向量补齐到同一长度，模拟真实 HCCL 的等长通信缓冲区。
        padded_vectors.push_back(pad_vector(vector, padding.padded_vocab_size));
    }

    const auto reduce_start = std::chrono::steady_clock::now();
    // Reduce 阶段：对所有 rank 的同一 token 位置做求和，得到全局词频向量。
    CountVector global_sum(padding.padded_vocab_size, 0);
    for (const auto& vector : padded_vectors) {
        for (std::size_t index = 0; index < vector.size(); ++index) {
            global_sum[index] += vector[index];
        }
    }

    // Scatter 阶段：把全局词频向量按 rank 数切成等长分片。
    std::vector<CountVector> shards;
    shards.reserve(static_cast<std::size_t>(world_size));
    for (int rank = 0; rank < world_size; ++rank) {
        const std::size_t start = static_cast<std::size_t>(rank) * padding.shard_size;
        shards.emplace_back(
            global_sum.begin() + static_cast<std::ptrdiff_t>(start),
            global_sum.begin() + static_cast<std::ptrdiff_t>(start + padding.shard_size));
    }
    const auto reduce_end = std::chrono::steady_clock::now();

    const auto gather_start = std::chrono::steady_clock::now();
    // AllGather 阶段：收集所有 rank 的分片，恢复完整全局词频向量。
    CountVector gathered;
    gathered.reserve(padding.padded_vocab_size);
    for (const auto& shard : shards) {
        gathered.insert(gathered.end(), shard.begin(), shard.end());
    }
    const auto gather_end = std::chrono::steady_clock::now();

    CommunicationResult result;
    result.padded_vocab_size = padding.padded_vocab_size;
    result.shard_size = padding.shard_size;
    result.reduced_shards = std::move(shards);
    result.gathered_count = std::move(gathered);
    result.reduce_scatter_ms =
        std::chrono::duration<double, std::milli>(reduce_end - reduce_start).count();
    result.all_gather_ms =
        std::chrono::duration<double, std::milli>(gather_end - gather_start).count();
    return result;
}

}  // namespace hccl_string_stats


### 4.4 HCCL集合通信后端

HCCL后端只在`ENABLE_HCCL=ON`时编译真实设备代码。每个进程初始化AscendCL、绑定`local_device`、创建执行流并加入HCCL通信域；随后申请完整输入向量、ReduceScatter结果分片和AllGather完整结果所需的Device内存。

通信阶段先把本地计数向量复制到Device，再执行`HcclReduceScatter`和`HcclAllGather`。每次集合通信后调用`aclrtSynchronizeStream`，保证计时结束前设备任务已经完成。最后把结果复制回Host，并释放通信器、Device内存、执行流和AscendCL资源。


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/src/hccl_backend.cpp
#include "hccl_backend.hpp"

#include <chrono>
#include <stdexcept>

#if defined(ENABLE_HCCL)

#include <acl/acl.h>
#include <hccl/hccl.h>

#include <cstdint>
#include <cstring>
#include <fstream>
#include <sstream>
#include <thread>

#endif

namespace hccl_string_stats {

#if defined(ENABLE_HCCL)

namespace {

void check_acl(aclError result, const char* expression) {
    if (result != ACL_ERROR_NONE) {
        std::ostringstream oss;
        oss << expression << " failed, acl error=" << static_cast<int>(result);
        throw std::runtime_error(oss.str());
    }
}

void check_hccl(HcclResult result, const char* expression) {
    if (result != HCCL_SUCCESS) {
        std::ostringstream oss;
        oss << expression << " failed, hccl error=" << static_cast<int>(result);
        throw std::runtime_error(oss.str());
    }
}

#define CHECK_ACL(expr) check_acl((expr), #expr)
#define CHECK_HCCL(expr) check_hccl((expr), #expr)

void write_root_info(const std::string& path, const HcclRootInfo& root_info) {
    // rank0 负责生成通信域 root info，并写入临时文件供其他 rank 读取。
    std::ofstream output(path, std::ios::binary | std::ios::trunc);
    if (!output) {
        throw std::runtime_error("failed to write HCCL root info file: " + path);
    }
    output.write(reinterpret_cast<const char*>(&root_info), sizeof(root_info));
}

HcclRootInfo read_root_info(const std::string& path) {
    HcclRootInfo root_info;
    std::memset(&root_info, 0, sizeof(root_info));

    // 其他 rank 可能先启动，因此这里轮询等待 rank0 写完 root info 文件。
    for (int retry = 0; retry < 600; ++retry) {
        std::ifstream input(path, std::ios::binary);
        if (input) {
            input.read(reinterpret_cast<char*>(&root_info), sizeof(root_info));
            if (input.gcount() == static_cast<std::streamsize>(sizeof(root_info))) {
                return root_info;
            }
        }
        std::this_thread::sleep_for(std::chrono::milliseconds(100));
    }

    throw std::runtime_error("timeout while waiting for HCCL root info file: " + path);
}

void* acl_malloc(std::size_t bytes) {
    void* ptr = nullptr;
    // HCCL 通信使用设备内存缓冲区，先通过 AscendCL 分配。
    CHECK_ACL(aclrtMalloc(&ptr, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    return ptr;
}

std::vector<CountVector> split_gathered_into_shards(
    const CountVector& gathered,
    std::size_t shard_size,
    int world_size) {
    // HCCL 只返回当前 rank 的 shard 和 AllGather 后的完整向量；
    // 为了统一结果展示，这里把完整向量重新切成每个 rank 的 shard。
    std::vector<CountVector> shards;
    shards.reserve(static_cast<std::size_t>(world_size));
    for (int rank = 0; rank < world_size; ++rank) {
        const std::size_t start = static_cast<std::size_t>(rank) * shard_size;
        shards.emplace_back(
            gathered.begin() + static_cast<std::ptrdiff_t>(start),
            gathered.begin() + static_cast<std::ptrdiff_t>(start + shard_size));
    }
    return shards;
}

}  // namespace

bool is_hccl_compiled() {
    return true;
}

CommunicationResult run_hccl_collectives(
    const CountVector& local_count,
    std::size_t vocab_size,
    const HcclRuntimeOptions& options) {
    if (options.world_size < 1) {
        throw std::invalid_argument("backend=hccl requires world_size >= 1");
    }
    if (options.rank < 0 || options.rank >= options.world_size) {
        throw std::invalid_argument("rank must be in [0, world_size)");
    }

    const PaddingInfo padding = compute_padding(vocab_size, options.world_size);
    const CountVector padded = pad_vector(local_count, padding.padded_vocab_size);
    CountVector host_shard(padding.shard_size, 0);
    CountVector host_gathered(padding.padded_vocab_size, 0);

    aclrtStream stream = nullptr;
    HcclComm comm = nullptr;
    void* send_device = nullptr;
    void* shard_device = nullptr;
    void* gathered_device = nullptr;

    try {
        CHECK_ACL(aclInit(nullptr));
        CHECK_ACL(aclrtSetDevice(options.local_device));
        CHECK_ACL(aclrtCreateStream(&stream));

        // rank0 生成 root info，其他 rank 读取同一份 root info 后加入同一个 HCCL 通信域。
        HcclRootInfo root_info;
        if (options.rank == 0) {
            CHECK_HCCL(HcclGetRootInfo(&root_info));
            write_root_info(options.root_info_file, root_info);
        } else {
            root_info = read_root_info(options.root_info_file);
        }

        CHECK_HCCL(HcclCommInitRootInfo(
            static_cast<std::uint32_t>(options.world_size),
            &root_info,
            static_cast<std::uint32_t>(options.rank),
            &comm));

        const std::size_t full_bytes = padding.padded_vocab_size * sizeof(Count);
        const std::size_t shard_bytes = padding.shard_size * sizeof(Count);
        send_device = acl_malloc(full_bytes);
        shard_device = acl_malloc(shard_bytes);
        gathered_device = acl_malloc(full_bytes);

        // 将本地计数向量从主机内存拷贝到设备内存，作为 ReduceScatter 输入。
        CHECK_ACL(aclrtMemcpy(
            send_device,
            full_bytes,
            padded.data(),
            full_bytes,
            ACL_MEMCPY_HOST_TO_DEVICE));

        const auto reduce_start = std::chrono::steady_clock::now();
        // HCCL ReduceScatter：先对各 rank 的计数向量逐项求和，再按 rank 切分结果。
        CHECK_HCCL(HcclReduceScatter(
            send_device,
            shard_device,
            static_cast<std::uint64_t>(padding.shard_size),
            HCCL_DATA_TYPE_INT64,
            HCCL_REDUCE_SUM,
            comm,
            stream));
        CHECK_ACL(aclrtSynchronizeStream(stream));
        const auto reduce_end = std::chrono::steady_clock::now();

        const auto gather_start = std::chrono::steady_clock::now();
        // HCCL AllGather：收集所有 rank 的分片，使每个 rank 都恢复完整全局计数向量。
        CHECK_HCCL(HcclAllGather(
            shard_device,
            gathered_device,
            static_cast<std::uint64_t>(padding.shard_size),
            HCCL_DATA_TYPE_INT64,
            comm,
            stream));
        CHECK_ACL(aclrtSynchronizeStream(stream));
        const auto gather_end = std::chrono::steady_clock::now();

        // 将通信结果拷贝回主机内存，供后续 Top-K、正确性校验和报告输出使用。
        CHECK_ACL(aclrtMemcpy(
            host_shard.data(),
            shard_bytes,
            shard_device,
            shard_bytes,
            ACL_MEMCPY_DEVICE_TO_HOST));
        CHECK_ACL(aclrtMemcpy(
            host_gathered.data(),
            full_bytes,
            gathered_device,
            full_bytes,
            ACL_MEMCPY_DEVICE_TO_HOST));

        CommunicationResult result;
        result.padded_vocab_size = padding.padded_vocab_size;
        result.shard_size = padding.shard_size;
        result.reduced_shards =
            split_gathered_into_shards(host_gathered, padding.shard_size, options.world_size);
        result.gathered_count = std::move(host_gathered);
        result.reduce_scatter_ms =
            std::chrono::duration<double, std::milli>(reduce_end - reduce_start).count();
        result.all_gather_ms =
            std::chrono::duration<double, std::milli>(gather_end - gather_start).count();

        // 正常路径主动释放 HCCL/ACL 资源；异常路径在 catch 中兜底释放。
        CHECK_HCCL(HcclCommDestroy(comm));
        comm = nullptr;
        CHECK_ACL(aclrtFree(send_device));
        send_device = nullptr;
        CHECK_ACL(aclrtFree(shard_device));
        shard_device = nullptr;
        CHECK_ACL(aclrtFree(gathered_device));
        gathered_device = nullptr;
        CHECK_ACL(aclrtDestroyStream(stream));
        stream = nullptr;
        CHECK_ACL(aclrtResetDevice(options.local_device));
        CHECK_ACL(aclFinalize());

        return result;
    } catch (...) {
        if (comm != nullptr) {
            HcclCommDestroy(comm);
        }
        if (send_device != nullptr) {
            aclrtFree(send_device);
        }
        if (shard_device != nullptr) {
            aclrtFree(shard_device);
        }
        if (gathered_device != nullptr) {
            aclrtFree(gathered_device);
        }
        if (stream != nullptr) {
            aclrtDestroyStream(stream);
        }
        aclrtResetDevice(options.local_device);
        aclFinalize();
        throw;
    }
}

#else

bool is_hccl_compiled() {
    return false;
}

CommunicationResult run_hccl_collectives(
    const CountVector&,
    std::size_t,
    const HcclRuntimeOptions&) {
    throw std::runtime_error(
        "HCCL backend is not compiled. Rebuild with cmake -DENABLE_HCCL=ON.");
}

#endif

}  // namespace hccl_string_stats


### 4.5 结果报告与主流程

结果报告模块输出实验配置、各Rank本地统计、ReduceScatter分片、全局Top-K、通信耗时、缓冲区规模和正确性检查。主流程根据`backend`选择模拟或HCCL路径，并统一执行全局向量去补齐、串行参考计算和逐元素校验。

真实HCCL运行时，所有Rank都参与通信，但只有Rank 0打印报告并写入结果文件，避免多个进程同时输出或覆盖同一个文件。


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/src/result_report.cpp
#include "result_report.hpp"

#include <algorithm>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <numeric>
#include <stdexcept>

namespace hccl_string_stats {
namespace {

std::string yes_no(bool value) {
    return value ? "PASS" : "FAIL";
}

}  // namespace

void print_experiment_config(
    const ExperimentConfig& config,
    const Vocabulary& vocabulary,
    const CommunicationResult& communication) {
    std::cout << "Experiment configuration\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::left << std::setw(22) << "backend" << ": " << backend_name(config.backend) << '\n';
    std::cout << std::left << std::setw(22) << "world_size" << ": " << config.world_size << '\n';
    if (config.backend == Backend::Hccl) {
        std::cout << std::left << std::setw(22) << "rank" << ": " << config.rank << '\n';
        std::cout << std::left << std::setw(22) << "local_device" << ": " << config.local_device << '\n';
    }
    std::cout << std::left << std::setw(22) << "corpus_mode" << ": " << config.corpus_mode << '\n';
    std::cout << std::left << std::setw(22) << "total_length" << ": " << config.total_length << '\n';
    std::cout << std::left << std::setw(22) << "vocab_size" << ": " << vocabulary.tokens.size() << '\n';
    std::cout << std::left << std::setw(22) << "padded_vocab_size" << ": " << communication.padded_vocab_size << '\n';
    std::cout << std::left << std::setw(22) << "shard_size" << ": " << communication.shard_size << '\n';
    std::cout << std::left << std::setw(22) << "seed" << ": " << config.seed << '\n';
    std::cout << std::string(88, '-') << '\n';
}

void print_rank_summary(const std::vector<RankLocalResult>& local_results, const Vocabulary& vocabulary) {
    std::cout << "\nRank local statistics\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::right << std::setw(4) << "rank"
              << std::setw(10) << "chars"
              << std::setw(17) << "nonzero_tokens"
              << "  top_local_tokens\n";
    std::cout << std::string(88, '-') << '\n';

    for (const auto& result : local_results) {
        // 展示每个 rank 的本地字符数、非零 token 数和局部 Top-K，便于观察数据分布差异。
        const int nonzero = static_cast<int>(
            std::count_if(result.count_vector.begin(), result.count_vector.end(), [](Count value) {
                return value > 0;
            }));
        const auto top_local = topk_tokens(result.count_vector, vocabulary, 5);
        std::cout << std::right << std::setw(4) << result.rank
                  << std::setw(10) << result.text.size()
                  << std::setw(17) << nonzero
                  << "  ";
        for (std::size_t i = 0; i < top_local.size(); ++i) {
            if (i > 0) {
                std::cout << ", ";
            }
            std::cout << display_token(top_local[i].first) << ':' << top_local[i].second;
        }
        std::cout << '\n';
    }
    std::cout << std::string(88, '-') << '\n';
}

void print_shard_summary(const std::vector<CountVector>& shards) {
    std::cout << "\nReduceScatter shard summary\n";
    std::cout << std::string(72, '-') << '\n';
    std::cout << std::right << std::setw(4) << "rank"
              << std::setw(12) << "shard_len"
              << std::setw(14) << "shard_sum"
              << std::setw(12) << "nonzero" << '\n';
    std::cout << std::string(72, '-') << '\n';
    for (std::size_t rank = 0; rank < shards.size(); ++rank) {
        // shard_sum 是该 rank 获得的全局词频分片总和，用于检查切片后的数据量分布。
        const Count shard_sum = std::accumulate(shards[rank].begin(), shards[rank].end(), Count{0});
        const int nonzero = static_cast<int>(
            std::count_if(shards[rank].begin(), shards[rank].end(), [](Count value) {
                return value > 0;
            }));
        std::cout << std::right << std::setw(4) << rank
                  << std::setw(12) << shards[rank].size()
                  << std::setw(14) << shard_sum
                  << std::setw(12) << nonzero << '\n';
    }
    std::cout << std::string(72, '-') << '\n';
}

void print_topk(const std::vector<std::pair<std::string, Count>>& top_tokens) {
    std::cout << "\nGlobal top-k tokens\n";
    std::cout << std::string(40, '-') << '\n';
    for (const auto& [token, count] : top_tokens) {
        std::cout << std::left << std::setw(8) << display_token(token)
                  << std::right << std::setw(8) << count << '\n';
    }
    std::cout << std::string(40, '-') << '\n';
}

void print_timing(const CommunicationResult& communication, Backend backend) {
    std::cout << "\nCommunication time in " << backend_name(backend) << " backend\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::left << std::setw(22) << "reduce_scatter_ms" << ": "
              << std::fixed << std::setprecision(4) << communication.reduce_scatter_ms << '\n';
    std::cout << std::left << std::setw(22) << "all_gather_ms" << ": "
              << std::fixed << std::setprecision(4) << communication.all_gather_ms << '\n';
    std::cout << std::string(88, '-') << '\n';
}

void print_buffer_estimate(const BufferEstimate& estimate) {
    std::cout << "\nEstimated buffer size, int64\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::left << std::setw(40) << "input_count_vector_bytes_per_rank"
              << ": " << std::right << std::setw(8) << estimate.input_count_vector_bytes_per_rank << " bytes\n";
    std::cout << std::left << std::setw(40) << "reduce_scatter_shard_bytes_per_rank"
              << ": " << std::right << std::setw(8) << estimate.reduce_scatter_shard_bytes_per_rank << " bytes\n";
    std::cout << std::left << std::setw(40) << "all_gather_result_bytes_per_rank"
              << ": " << std::right << std::setw(8) << estimate.all_gather_result_bytes_per_rank << " bytes\n";
    std::cout << std::left << std::setw(40) << "all_ranks_input_bytes_total"
              << ": " << std::right << std::setw(8) << estimate.all_ranks_input_bytes_total << " bytes\n";
    std::cout << std::string(88, '-') << '\n';
}

void print_correctness(bool correct, const std::vector<std::size_t>& mismatch_indices, double load_imbalance) {
    std::cout << "\nCorrectness check\n";
    std::cout << std::string(88, '-') << '\n';
    std::cout << std::left << std::setw(22) << "result" << ": " << yes_no(correct) << '\n';
    std::cout << std::left << std::setw(22) << "mismatch_count" << ": " << mismatch_indices.size() << '\n';
    std::cout << std::left << std::setw(22) << "load_imbalance" << ": "
              << std::fixed << std::setprecision(3) << load_imbalance << '\n';
    if (!mismatch_indices.empty()) {
        std::cout << std::left << std::setw(22) << "mismatch_preview" << ": ";
        const std::size_t preview_size = std::min<std::size_t>(10, mismatch_indices.size());
        for (std::size_t i = 0; i < preview_size; ++i) {
            if (i > 0) {
                std::cout << ", ";
            }
            std::cout << mismatch_indices[i];
        }
        std::cout << '\n';
    }
    std::cout << std::string(88, '-') << '\n';
}

void write_result_summary(
    const std::string& output_path,
    const ExperimentConfig& config,
    const CommunicationResult& communication,
    bool correct,
    const std::vector<std::size_t>& mismatch_indices,
    double load_imbalance,
    const std::vector<std::pair<std::string, Count>>& top_tokens) {
    // 写入精简结果文件，方便 Notebook、实验报告或后续自动化脚本复用。
    std::ofstream output(output_path);
    if (!output) {
        throw std::runtime_error("failed to open result file: " + output_path);
    }

    output << "backend: " << backend_name(config.backend) << '\n';
    output << "world_size: " << config.world_size << '\n';
    output << "corpus_mode: " << config.corpus_mode << '\n';
    output << "total_length: " << config.total_length << '\n';
    output << "padded_vocab_size: " << communication.padded_vocab_size << '\n';
    output << "shard_size: " << communication.shard_size << '\n';
    output << "reduce_scatter_ms: " << communication.reduce_scatter_ms << '\n';
    output << "all_gather_ms: " << communication.all_gather_ms << '\n';
    output << "result: " << yes_no(correct) << '\n';
    output << "mismatch_count: " << mismatch_indices.size() << '\n';
    output << "load_imbalance: " << load_imbalance << '\n';
    output << "top_tokens:\n";
    for (const auto& [token, count] : top_tokens) {
        output << "  " << display_token(token) << ": " << count << '\n';
    }
}

}  // namespace hccl_string_stats


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/src/main.cpp
#include <algorithm>
#include <cstdlib>
#include <exception>
#include <iostream>
#include <numeric>
#include <stdexcept>
#include <string>
#include <vector>

#include "collectives.hpp"
#include "corpus.hpp"
#include "experiment_config.hpp"
#include "hccl_backend.hpp"
#include "result_report.hpp"
#include "string_statistics.hpp"
#include "vocab.hpp"

namespace hccl_string_stats {
namespace {

int read_env_int(const char* name, int fallback) {
    const char* value = std::getenv(name);
    if (value == nullptr || std::string(value).empty()) {
        return fallback;
    }
    return std::stoi(value);
}

void print_usage(const char* program) {
    std::cout
        << "Usage: " << program << " [options]\n"
        << "\n"
        << "Options:\n"
        << "  --backend simulate|hccl       Backend to run, default: simulate\n"
        << "  --world-size N                Number of logical ranks, default: 4 or WORLD_SIZE\n"
        << "  --rank N                      Current rank for backend=hccl, default: RANK\n"
        << "  --local-device N              Ascend device id for backend=hccl, default: LOCAL_RANK\n"
        << "  --corpus-mode MODE            uniform, skewed, or rank_biased\n"
        << "  --total-length N              Generated text length\n"
        << "  --seed N                      Deterministic random seed\n"
        << "  --top-k N                     Number of global tokens to print\n"
        << "  --root-info-file PATH         File used to exchange HCCL root info\n"
        << "  --output PATH                 Write a compact result summary\n"
        << "  --help                        Show this message\n";
}

struct ParsedArgs {
    ExperimentConfig config;
    std::string output_path;
};

ParsedArgs parse_args(int argc, char** argv) {
    ParsedArgs parsed;
    // HCCL 启动脚本会为每个进程设置 WORLD_SIZE/RANK/LOCAL_RANK。
    // 命令行参数仍然可以覆盖这些环境变量，便于单独调试。
    parsed.config.world_size = read_env_int("WORLD_SIZE", parsed.config.world_size);
    parsed.config.rank = read_env_int("RANK", parsed.config.rank);
    parsed.config.local_device = read_env_int("LOCAL_RANK", parsed.config.rank);

    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto require_value = [&](const std::string& option) -> std::string {
            if (i + 1 >= argc) {
                throw std::invalid_argument(option + " requires a value");
            }
            return argv[++i];
        };

        if (arg == "--help" || arg == "-h") {
            print_usage(argv[0]);
            std::exit(0);
        } else if (arg == "--backend") {
            parsed.config.backend = parse_backend(require_value(arg));
        } else if (arg == "--world-size") {
            parsed.config.world_size = std::stoi(require_value(arg));
        } else if (arg == "--rank") {
            parsed.config.rank = std::stoi(require_value(arg));
        } else if (arg == "--local-device") {
            parsed.config.local_device = std::stoi(require_value(arg));
        } else if (arg == "--corpus-mode") {
            parsed.config.corpus_mode = require_value(arg);
        } else if (arg == "--total-length") {
            parsed.config.total_length = std::stoi(require_value(arg));
        } else if (arg == "--seed") {
            parsed.config.seed = static_cast<std::uint32_t>(std::stoul(require_value(arg)));
        } else if (arg == "--top-k") {
            parsed.config.top_k = std::stoi(require_value(arg));
        } else if (arg == "--root-info-file") {
            parsed.config.root_info_file = require_value(arg);
        } else if (arg == "--output") {
            parsed.output_path = require_value(arg);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }

    if (parsed.output_path.empty()) {
        parsed.output_path = parsed.config.backend == Backend::Simulate
                                 ? "results/simulate_latest.txt"
                                 : "results/hccl_rank0_latest.txt";
    }

    validate_config(parsed.config);
    return parsed;
}

double compute_load_imbalance(const std::vector<RankLocalResult>& local_results) {
    if (local_results.empty()) {
        return 0.0;
    }
    // 这里的负载不均衡只按每个 rank 的文本长度估算，
    // 用于说明数据切分是否均匀，不代表通信耗时。
    std::vector<std::size_t> lengths;
    lengths.reserve(local_results.size());
    for (const auto& result : local_results) {
        lengths.push_back(result.text.size());
    }
    const double total = static_cast<double>(
        std::accumulate(lengths.begin(), lengths.end(), std::size_t{0}));
    const double average = total / static_cast<double>(lengths.size());
    if (average == 0.0) {
        return 0.0;
    }
    const auto max_iter = std::max_element(lengths.begin(), lengths.end());
    return static_cast<double>(*max_iter) / average;
}

std::vector<RankLocalResult> build_all_rank_results(
    const std::vector<std::string>& rank_texts,
    const Vocabulary& vocabulary) {
    std::vector<RankLocalResult> local_results;
    local_results.reserve(rank_texts.size());
    for (std::size_t rank = 0; rank < rank_texts.size(); ++rank) {
        local_results.push_back(RankLocalResult{
            static_cast<int>(rank),
            rank_texts[rank],
            count_local_text(rank_texts[rank], vocabulary),
        });
    }
    return local_results;
}

/* CPU模拟实现版本 */
int run_simulate(const ExperimentConfig& config, const std::string& output_path) {
    // simulate 后端在一个进程里构造所有 rank 的本地数据，
    // 便于在没有 HCCL 环境时验证 ReduceScatter/AllGather 的数据语义。
    const Vocabulary vocabulary = build_vocabulary();
    const std::string full_text = generate_corpus(config.corpus_mode, config.total_length, config.seed);
    const std::vector<std::string> rank_texts = split_text_by_rank(full_text, config.world_size);
    const std::vector<RankLocalResult> local_results = build_all_rank_results(rank_texts, vocabulary);

    // 提取每个 rank 的本地计数向量，作为模拟集合通信的输入。
    std::vector<CountVector> local_vectors;
    local_vectors.reserve(local_results.size());
    for (const auto& result : local_results) {
        local_vectors.push_back(result.count_vector);
    }

    CommunicationResult communication = run_simulated_collectives(local_vectors, config.world_size);
    // AllGather 恢复出的向量包含 padding 位置，和真实词表比较前要去掉补齐部分。
    const CountVector global_count = unpad_vector(communication.gathered_count, vocabulary.tokens.size());
    const CountVector reference_count = serial_reference(full_text, vocabulary);
    const auto [correct, mismatch_indices] = verify_result(global_count, reference_count);
    const auto top_tokens = topk_tokens(global_count, vocabulary, config.top_k);
    const BufferEstimate estimate =
        estimate_bytes(communication.padded_vocab_size, communication.shard_size, config.world_size);
    const double load_imbalance = compute_load_imbalance(local_results);

    print_experiment_config(config, vocabulary, communication);
    print_rank_summary(local_results, vocabulary);
    print_shard_summary(communication.reduced_shards);
    print_topk(top_tokens);
    print_timing(communication, config.backend);
    print_buffer_estimate(estimate);
    print_correctness(correct, mismatch_indices, load_imbalance);
    write_result_summary(output_path, config, communication, correct, mismatch_indices, load_imbalance, top_tokens);

    return correct ? 0 : 2;
}

/* HCCL集合通信实现 */
int run_hccl(const ExperimentConfig& config, const std::string& output_path) {
    if (!is_hccl_compiled()) {
        throw std::runtime_error(
            "backend=hccl requires rebuilding with -DENABLE_HCCL=ON and Ascend Toolkit libraries");
    }

    // HCCL 后端中，每个进程只统计自己 rank 对应的文本片段。
    // 所有进程使用相同 seed 生成同一份全局语料，从而保证切分结果一致。

    //字符集合构建
    const Vocabulary vocabulary = build_vocabulary();

    // 步骤一：实验文本生成与数据划分。
    const std::string full_text = generate_corpus(config.corpus_mode, config.total_length, config.seed);

    // 步骤二：计数向量长度对齐与结果分片规划
    const std::vector<std::string> rank_texts = split_text_by_rank(full_text, config.world_size);

    // 步骤三：本地计数向量统计。
    const CountVector local_count = count_local_text(rank_texts[static_cast<std::size_t>(config.rank)], vocabulary);

    const HcclRuntimeOptions options{
        config.rank,
        config.world_size,
        config.local_device,
        config.root_info_file,
    };

    // 步骤四：HCCL 集合通信执行。
    CommunicationResult communication = run_hccl_collectives(local_count, vocabulary.tokens.size(), options);

    // 步骤五：结果校验与实验分析。通信结果与串行参考结果对比，验证 ReduceScatter + AllGather 没有丢失或错位。
    const CountVector global_count = unpad_vector(communication.gathered_count, vocabulary.tokens.size());
    const CountVector reference_count = serial_reference(full_text, vocabulary);
    const auto [correct, mismatch_indices] = verify_result(global_count, reference_count);
    const auto top_tokens = topk_tokens(global_count, vocabulary, config.top_k);
    const BufferEstimate estimate =
        estimate_bytes(communication.padded_vocab_size, communication.shard_size, config.world_size);
    const std::vector<RankLocalResult> local_results = build_all_rank_results(rank_texts, vocabulary);
    const double load_imbalance = compute_load_imbalance(local_results);

    // 多进程运行时只让 rank0 打印和写结果，避免多个 rank 同时输出造成日志混杂。
    if (config.rank == 0) {
        print_experiment_config(config, vocabulary, communication);
        print_rank_summary(local_results, vocabulary);
        print_shard_summary(communication.reduced_shards);
        print_topk(top_tokens);
        print_timing(communication, config.backend);
        print_buffer_estimate(estimate);
        print_correctness(correct, mismatch_indices, load_imbalance);
        write_result_summary(output_path, config, communication, correct, mismatch_indices, load_imbalance, top_tokens);
    }

    return correct ? 0 : 2;
}

}  // namespace
}  // namespace hccl_string_stats

int main(int argc, char** argv) {
    try {
        const auto parsed = hccl_string_stats::parse_args(argc, argv);
        if (parsed.config.backend == hccl_string_stats::Backend::Simulate) {
            return hccl_string_stats::run_simulate(parsed.config, parsed.output_path);
        }
        return hccl_string_stats::run_hccl(parsed.config, parsed.output_path);
    } catch (const std::exception& exc) {
        std::cerr << "error: " << exc.what() << '\n';
        return 1;
    }
}


### 4.6 核心源码检查

检查所有头文件和源文件是否已经写入。全部显示`OK`时，说明C++工程源码完整，可以进入构建和运行阶段。


In [ ]:
from pathlib import Path

required_files = [
    "include/experiment_config.hpp",
    "include/vocab.hpp",
    "include/corpus.hpp",
    "include/string_statistics.hpp",
    "include/collectives.hpp",
    "include/hccl_backend.hpp",
    "include/result_report.hpp",
    "src/experiment_config.cpp",
    "src/vocab.cpp",
    "src/corpus.cpp",
    "src/string_statistics.cpp",
    "src/simulate_collectives.cpp",
    "src/hccl_backend.cpp",
    "src/result_report.cpp",
    "src/main.cpp",
]

for relative_path in required_files:
    path = WORK_DIR / relative_path
    print(f"{relative_path}: {'OK' if path.exists() else 'MISSING'}")


---
## 5. 结果验证与性能分析

核心源码准备完成后，本节写入CMake构建配置和运行脚本，先运行`simulate`后端验证数据结构和集合通信语义，再自动检测CANN环境和可见NPU数量。检测到两张及以上NPU时运行真实双Rank HCCL；检测到一张NPU时尝试真实单Rank HCCL；没有可用NPU或真实HCCL启动失败时自动回退到simulate双Rank。程序将结果保存到`results`目录，便于Notebook读取和分析。



### 5.1 工程构建

`CMakeLists.txt`使用C++17构建统一可执行程序。默认关闭HCCL，使普通CPU环境也能编译模拟后端；运行真实HCCL时，构建脚本传入`-DENABLE_HCCL=ON`，并链接CANN中的`hccl`与`ascendcl`库。


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

project(hccl_string_statistics_experiment LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

option(ENABLE_HCCL "Build with Ascend HCCL backend" OFF)
set(ASCEND_TOOLKIT_HOME "" CACHE PATH "Ascend toolkit root, for example /usr/local/Ascend/ascend-toolkit/latest")

add_executable(hccl_string_statistics
    src/main.cpp
    src/experiment_config.cpp
    src/vocab.cpp
    src/corpus.cpp
    src/string_statistics.cpp
    src/simulate_collectives.cpp
    src/hccl_backend.cpp
    src/result_report.cpp
)

target_include_directories(hccl_string_statistics PRIVATE include)

if (MSVC)
    target_compile_options(hccl_string_statistics PRIVATE /W4)
else()
    target_compile_options(hccl_string_statistics PRIVATE -Wall -Wextra -pedantic)
endif()

if (ENABLE_HCCL)
    target_compile_definitions(hccl_string_statistics PRIVATE ENABLE_HCCL=1)

    if (NOT ASCEND_TOOLKIT_HOME)
        if (DEFINED ENV{ASCEND_TOOLKIT_HOME})
            set(ASCEND_TOOLKIT_HOME "$ENV{ASCEND_TOOLKIT_HOME}")
        elseif (EXISTS "/usr/local/Ascend/ascend-toolkit/latest")
            set(ASCEND_TOOLKIT_HOME "/usr/local/Ascend/ascend-toolkit/latest")
        else()
            message(FATAL_ERROR "ENABLE_HCCL=ON requires ASCEND_TOOLKIT_HOME")
        endif()
    endif()

    target_include_directories(hccl_string_statistics PRIVATE
        "${ASCEND_TOOLKIT_HOME}/include"
    )

    find_library(HCCL_LIBRARY
        NAMES hccl
        HINTS
            "${ASCEND_TOOLKIT_HOME}/lib64"
            "${ASCEND_TOOLKIT_HOME}/hccl/lib64"
            "${ASCEND_TOOLKIT_HOME}/runtime/lib64/stub"
    )
    find_library(ASCENDCL_LIBRARY
        NAMES ascendcl
        HINTS
            "${ASCEND_TOOLKIT_HOME}/lib64"
            "${ASCEND_TOOLKIT_HOME}/runtime/lib64"
            "${ASCEND_TOOLKIT_HOME}/runtime/lib64/stub"
    )

    if (NOT HCCL_LIBRARY)
        message(FATAL_ERROR "Could not find libhccl. Set ASCEND_TOOLKIT_HOME or HCCL_LIBRARY.")
    endif()
    if (NOT ASCENDCL_LIBRARY)
        message(FATAL_ERROR "Could not find libascendcl. Set ASCEND_TOOLKIT_HOME or ASCENDCL_LIBRARY.")
    endif()

    target_link_libraries(hccl_string_statistics PRIVATE
        "${HCCL_LIBRARY}"
        "${ASCENDCL_LIBRARY}"
    )
endif()


### 5.2 运行脚本

`run_simulate.sh`负责构建并运行CPU模拟后端。`run_hccl.sh`负责清理旧RootInfo和Rank日志、构建HCCL版本、为每个Rank设置`WORLD_SIZE`、`RANK`和`LOCAL_RANK`，并行启动多个进程，最后输出Rank 0日志。


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/scripts/run_simulate.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
PROJECT_DIR="$(cd "${SCRIPT_DIR}/.." && pwd)"

cd "${PROJECT_DIR}"
mkdir -p results

cmake -S . -B build -DENABLE_HCCL=OFF
cmake --build build

./build/hccl_string_statistics \
  --backend simulate \
  --output results/simulate_latest.txt \
  "$@"


In [ ]:
%%writefile src/03.01_inclass_hccl_string_statistics/scripts/run_hccl.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
PROJECT_DIR="$(cd "${SCRIPT_DIR}/.." && pwd)"

WORLD_SIZE="${1:-4}"
if [[ $# -gt 0 ]]; then
  shift
fi

ROOT_INFO_FILE="${ROOT_INFO_FILE:-/tmp/hccl_string_statistics_root.info}"
BUILD_DIR="${BUILD_DIR:-build-hccl}"

cd "${PROJECT_DIR}"
mkdir -p results
rm -f "${ROOT_INFO_FILE}"
rm -f results/hccl_rank_*.log results/hccl_rank0_latest.txt

cmake -S . -B "${BUILD_DIR}" -DENABLE_HCCL=ON
cmake --build "${BUILD_DIR}"

pids=()
for rank in $(seq 0 $((WORLD_SIZE - 1))); do
  (
    export RANK="${rank}"
    export WORLD_SIZE="${WORLD_SIZE}"
    export LOCAL_RANK="${rank}"
    "./${BUILD_DIR}/hccl_string_statistics" \
      --backend hccl \
      --world-size "${WORLD_SIZE}" \
      --rank "${rank}" \
      --local-device "${rank}" \
      --root-info-file "${ROOT_INFO_FILE}" \
      --output results/hccl_rank0_latest.txt \
      "$@" \
      > "results/hccl_rank_${rank}.log" 2>&1
  ) &
  pids+=("$!")
done

status=0
for pid in "${pids[@]}"; do
  if ! wait "${pid}"; then
    status=1
  fi
done

if [[ -f results/hccl_rank_0.log ]]; then
  cat results/hccl_rank_0.log
fi

exit "${status}"


In [ ]:
!chmod +x src/03.01_inclass_hccl_string_statistics/scripts/run_simulate.sh
!chmod +x src/03.01_inclass_hccl_string_statistics/scripts/run_hccl.sh
!find src/03.01_inclass_hccl_string_statistics -maxdepth 3 -type f | sort


### 5.3 运行simulate后端

先运行双Rank模拟实验，验证字符表、文本切分、本地计数、向量补齐、ReduceScatter语义、AllGather语义和串行参考校验。该命令不依赖CANN和NPU，会完成CPU工程构建和运行。


In [ ]:
!cd src/03.01_inclass_hccl_string_statistics && bash scripts/run_simulate.sh --world-size 2 --corpus-mode rank_biased --total-length 20000 --seed 2026 --top-k 12 | tee results/simulate_notebook.txt


### 5.4 自动选择HCCL或simulate后端

本单元根据当前环境自动选择运行方式：两张及以上可见NPU使用真实HCCL双Rank，一张可见NPU使用真实HCCL单Rank，没有NPU则使用simulate双Rank。若HCCL环境或运行过程失败，会打印失败原因并回退到simulate双Rank，保证Notebook能够继续完成正确性验证。

单Rank HCCL只能验证真实HCCL初始化、设备内存和集合通信API调用，不代表跨设备双Rank通信；只有检测到至少两张NPU并且真实HCCL运行成功时，才记录为双Rank HCCL结果。



In [ ]:
import os
import re
import shlex
import shutil
import subprocess
from pathlib import Path


def find_cann_env_script():
    """Find a CANN set_env.sh without assuming one fixed installation path."""
    candidates = []
    for name in ["CANN_ENV_SCRIPT", "ASCEND_INSTALL_PATH", "ASCEND_TOOLKIT_HOME", "ASCEND_HOME_PATH"]:
        value = os.environ.get(name)
        if not value:
            continue
        path = Path(value)
        if path.name == "set_env.sh":
            candidates.append(path)
        candidates.extend([
            path / "set_env.sh",
            path / "latest" / "set_env.sh",
            path / f"{os.uname().machine}-linux" / "set_env.sh",
        ])

    roots = [Path("/opt/conda/Ascend"), Path("/usr/local/Ascend"), Path("/opt/Ascend"), Path.home() / "Ascend"]
    for root in roots:
        if not root.exists():
            continue
        find_result = subprocess.run(
            ["find", str(root), "-type", "f", "-name", "set_env.sh", "-print", "-quit"],
            capture_output=True,
            text=True,
            check=False,
        )
        if find_result.stdout.strip():
            candidates.append(Path(find_result.stdout.strip()))

    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return None


def detect_visible_npu_count():
    """Detect devices visible to this process, respecting container visibility variables."""
    for name in ["ASCEND_RT_VISIBLE_DEVICES", "ASCEND_VISIBLE_DEVICES"]:
        value = os.environ.get(name)
        if value is None or not value.strip():
            continue
        if value.strip() in {"-1", "none", "None"}:
            return 0
        return len([item for item in re.split(r"[,;]", value) if item.strip() and item.strip() != "-1"])

    npu_smi = shutil.which("npu-smi")
    if npu_smi:
        result = subprocess.run([npu_smi, "info", "-l"], capture_output=True, text=True, check=False)
        text = result.stdout + result.stderr
        for pattern in [
            r"Total\s+number\s+of\s+NPU\s*:\s*(\d+)",
            r"Total\s+NPU\s*(?:number|count)?\s*:\s*(\d+)",
        ]:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                return int(match.group(1))
        ids = set(re.findall(r"NPU\s*ID\s*:\s*(\d+)", text, re.IGNORECASE))
        if ids:
            return len(ids)

    device_path = Path("/dev")
    if device_path.exists():
        return sum(1 for item in device_path.glob("davinci*") if re.fullmatch(r"davinci\d+", item.name))
    return 0


CANN_ENV_SCRIPT = find_cann_env_script()
NPU_COUNT = detect_visible_npu_count()


def run_backend(backend, world_size, output_name, console_name, extra_args):
    """Run one backend and save the complete console output for later inspection."""
    args = list(extra_args) + ["--output", output_name]
    if backend == "hccl":
        if CANN_ENV_SCRIPT is None:
            print("HCCL skipped: CANN set_env.sh was not found.")
            return 2
        shell_command = (
            f"source {shlex.quote(str(CANN_ENV_SCRIPT))} && "
            f"bash scripts/run_hccl.sh {world_size} "
            f"{' '.join(shlex.quote(item) for item in args)}"
        )
        command = ["bash", "-lc", shell_command]
    else:
        command = ["bash", "scripts/run_simulate.sh"] + args

    completed = subprocess.run(
        command,
        cwd=WORK_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    print(completed.stdout)
    (WORK_DIR / "results" / console_name).write_text(completed.stdout, encoding="utf-8")
    return completed.returncode


print("detected_npu_count:", NPU_COUNT)
print("cann_env_script:", CANN_ENV_SCRIPT if CANN_ENV_SCRIPT else "not found")

if NPU_COUNT >= 2 and CANN_ENV_SCRIPT is not None:
    selected_backend = "hccl"
    selected_world_size = 2
    fallback_reason = ""
elif NPU_COUNT == 1 and CANN_ENV_SCRIPT is not None:
    selected_backend = "hccl"
    selected_world_size = 1
    fallback_reason = "only one visible NPU; this is a single-rank HCCL validation"
else:
    selected_backend = "simulate"
    selected_world_size = 2
    fallback_reason = "fewer than one usable NPU or CANN environment unavailable"

print("selected_backend:", selected_backend)
print("selected_world_size:", selected_world_size)
print("fallback_reason:", fallback_reason or "none")

common_args = [
    "--corpus-mode", "rank_biased",
    "--total-length", "20000",
    "--seed", "2026",
    "--top-k", "12",
]
actual_backend = selected_backend
actual_world_size = selected_world_size
return_code = run_backend(
    selected_backend,
    selected_world_size,
    "results/hccl_auto_latest.txt" if selected_backend == "hccl" else "results/simulate_auto_latest.txt",
    "hccl_auto_notebook.txt" if selected_backend == "hccl" else "simulate_auto_notebook.txt",
    common_args,
)

if selected_backend == "hccl" and return_code != 0:
    print("HCCL execution failed; falling back to simulate with world_size=2.")
    print("fallback_reason: HCCL command returned", return_code)
    actual_backend = "simulate"
    actual_world_size = 2
    return_code = run_backend(
        "simulate",
        2,
        "results/simulate_auto_latest.txt",
        "simulate_auto_notebook.txt",
        common_args,
    )

print("actual_backend:", actual_backend)
print("actual_world_size:", actual_world_size)
print("run_return_code:", return_code)



### 5.5 查看实验结果

模拟后端将精简结果写入`simulate_latest.txt`或自动回退时的`simulate_auto_latest.txt`；真实HCCL后端由Rank 0写入`hccl_rank0_latest.txt`或自动选择时的`hccl_auto_latest.txt`。下面读取当前目录中可用的结果文件，并同时显示自动选择过程的后端与world size。



In [ ]:
from pathlib import Path

print("detected_npu_count:", globals().get("NPU_COUNT", "not detected"))
print("actual_backend:", globals().get("actual_backend", "run 5.4 first"))
print("actual_world_size:", globals().get("actual_world_size", "run 5.4 first"))

result_paths = [
    WORK_DIR / "results" / "simulate_latest.txt",
    WORK_DIR / "results" / "simulate_auto_latest.txt",
    WORK_DIR / "results" / "hccl_rank0_latest.txt",
    WORK_DIR / "results" / "hccl_auto_latest.txt",
]

for path in result_paths:
    print("=", path)
    if path.exists():
        print(path.read_text(encoding="utf-8", errors="ignore")[:4000])
    else:
        print("not found")



### 5.6 正确性验证

正确性验证重点观察`result`和`mismatch_count`。`result=PASS`且`mismatch_count=0`表示AllGather恢复出的全局计数向量与CPU串行扫描完整文本得到的参考向量逐元素一致。

还应检查以下关系：

1. 当实际world size为2时，`padded_vocab_size=46`、`shard_size=23`；当实际world size为1时，`padded_vocab_size=45`、`shard_size=45`；
2. 所有ReduceScatter分片的`shard_sum`之和等于`total_length`；
3. 全局Top-K字符的计数与串行参考结果一致；
4. `load_imbalance=1.000`表示各Rank持有的文本长度相同



In [ ]:
from pathlib import Path

def read_summary(path):
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        if ":" in line and not line.startswith("  "):
            key, value = line.split(":", 1)
            values[key.strip()] = value.strip()
    return values

print("detected_npu_count:", globals().get("NPU_COUNT", "not detected"))
print("actual_backend:", globals().get("actual_backend", "run 5.4 first"))
print("actual_world_size:", globals().get("actual_world_size", "run 5.4 first"))

filenames = [
    "simulate_latest.txt",
    "simulate_auto_latest.txt",
    "hccl_rank0_latest.txt",
    "hccl_auto_latest.txt",
]

for filename in filenames:
    path = WORK_DIR / "results" / filename
    values = read_summary(path)
    print("=", filename)
    if not values:
        print("not found")
        continue
    for key in [
        "backend",
        "world_size",
        "padded_vocab_size",
        "shard_size",
        "result",
        "mismatch_count",
        "load_imbalance",
    ]:
        print(f"{key}: {values.get(key, 'MISSING')}")


### 5.7 通信与缓冲区分析

双Rank配置下，每个Rank输入46个`int64`元素，共368字节；ReduceScatter后每个Rank保留23个元素，共184字节；AllGather后每个Rank恢复46个元素，共368字节。单Rank配置下，补齐后词表长度为45，输入、分片和AllGather结果均为45个`int64`，通信域中没有跨设备数据交换。

`reduce_scatter_ms`统计HCCL ReduceScatter调用和执行流同步时间，`all_gather_ms`统计HCCL AllGather调用和执行流同步时间。由于缓冲区较小，测量值主要体现集合通信固定开销、Rank同步、设备调度和流同步，不适合直接换算有效带宽。若要比较通信性能，应在同一通信域中先预热，再重复执行多次并统计中位数、平均值和标准差。

本实验更应关注通信流程是否正确，以及从变长字符串到定长数值向量的转换是否满足集合通信的数据布局要求。



---
## 6. 实验总结

本实验按照实验概述、环境准备、问题分析、核心程序开发和结果验证与性能分析五个阶段，实现了基于HCCL的分布式字符串词频统计。

* 字符表建立了字符与Token编号之间的稳定映射，使变长文本能够转换为定长`int64`计数向量。
* 全局文本按照Rank连续切分，每个进程只统计自己的本地文本片段，并生成与词表长度一致的局部贡献向量。
* 向量补齐保证计数向量长度能够被`world_size`整除，使ReduceScatter可以向每个Rank分发等长结果分片。
* ReduceScatter完成各Rank局部计数向量的逐元素求和与全局结果切分，AllGather收集所有结果分片并恢复完整全局词频向量。
* `simulate`后端用于在普通CPU环境中验证数据结构和通信语义，`hccl`后端用于双NPU服务器上的真实多进程集合通信。
* 结果验证通过CPU串行参考向量、`mismatch_count`和全局Top-K字符检查通信结果；缓冲区分析说明了完整输入向量、结果分片和AllGather完整结果之间的存储关系。

通过本实验，可以理解不规则字符串数据进入集合通信前的数据结构转换过程，掌握本地统计、向量补齐、ReduceScatter、AllGather和结果校验的完整流程，并具备分析分布式字符串统计正确性与通信开销的基本能力。
